# Eyes on the road

Import the main libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.layers.experimental import preprocessing
import keras_tuner as kt
import matplotlib.pyplot as plt

Define configuration constants

In [2]:
INPUT_SHAPE = (224, 224, 3)
DATA_DIR = './datasets/eyes_on_the_road'
BATCH_SIZE = 32
IMG_SIZE = (180, 180)
VALIDATION_SPLIT = 0.2

Load image dataset folder into datasets

In [3]:
# https://www.tensorflow.org/api_docs/python/tf/keras/utils/image_dataset_from_directory
(train_ds, val_ds) = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='int',
    validation_split=VALIDATION_SPLIT,
    subset='both',
    color_mode='grayscale',
    seed=2,
    shuffle=0,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE)

Found 40 files belonging to 2 classes.
Using 32 files for training.
Using 8 files for validation.


Print some pre-process information

In [4]:
class_names = train_ds.class_names
num_classes = len(class_names)
print(class_names)

['awake', 'drowsy']


Print the first 10 images to check the preprocessing.

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.show()

We can do some data augmentation and integrate it into the deep neural network.

In [ ]:
layers_data_augmentation = tf.keras.Sequential(
    [
        preprocessing.RandomFlip("horizontal"),
        preprocessing.RandomRotation(0.3, fill_mode='constant'),
        preprocessing.RandomZoom(0.2),
        tf.keras.layers.RandomBrightness(factor=0.2)
    ]
)

As we are going to use hyperparameters calculations must define de function model_builder taht returns a compiled model. We will be using this function for tunning the model parameters.

In [5]:
def model_builder(hp):
    model = tf.keras.Sequential();
    # model.add(layers_data_augmentation)
    model.add(tf.keras.layers.Flatten(input_shape=IMG_SIZE))

    hp_activation = hp.Choice('activation', values=['relu', 'tanh'])
    hp_layer_1_nodes = hp.Int('layer_1_nodes', min_value=32, max_value=128, step=16)
    hp_layer_2_nodes = hp.Int('layer_2_nodes', min_value=32, max_value=128, step=16)
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.add(tf.keras.layers.Dense(units=hp_layer_1_nodes, activation=hp_activation))
    model.add(tf.keras.layers.Dense(units=hp_layer_2_nodes, activation=hp_activation))
    model.add(tf.keras.layers.Dense(num_classes, activation='softmax'))

    model.compile(
        # optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=hp_learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )
    return model

Now we can define the tuner that will be in charge of tune al parameters to find the best model configuration.

In [6]:
tuner = kt.Hyperband(model_builder,
     objective='accuracy',  # duda: deberíamos de utilizar 'val_accuracy' (validación) pero no funciona.
     max_epochs=10,
     factor=3,
     directory='hyper_parameters',
     project_name='eyes_on_the_road'
     )

Configure the callbacks.

In [7]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=3)

Search for the best model parameters.

In [9]:
tuner.search(train_ds, epochs=10, callbacks=[early_stop])

INFO:tensorflow:Oracle triggered exit


Now we can print the results of the best model configuration:

In [ ]:
best_hps= tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.get_best_models(num_models=1)[0]
loss, accuracy = best_model.evaluate(val_ds)
print("Accuracy: {:.2f}%".format(accuracy * 100))